# HEAT metrics validation (M2)

Validate Blast + HitTrax day-slice aggregates for one athlete **before** trusting the PDF.

Default subject: **Jason Peele**, assessment **#3** (`2026-03-08`).

## Setup
1. Cloud SQL Auth Proxy listening on `127.0.0.1:3306`
2. This notebook auto-loads `DB_USER` / `DB_PASS` from GCP Secret Manager
3. Kernel: repo `.venv`. Optional: `pip install matplotlib`

### Cell — imports & DB config

Puts `backend/` on `sys.path`, sets host/port/database, and loads `db` + `report_metrics`.

**Credentials:** auto-loads from Secret Manager via `gcloud.cmd` (resolved by full path — Jupyter often lacks shell PATH).

In [1]:
import os
import sys
import shutil
import subprocess
import importlib
from pathlib import Path

ROOT = Path.cwd().resolve()
if (ROOT / "backend").is_dir():
    BACKEND = ROOT / "backend"
elif ROOT.name == "notebooks":
    BACKEND = ROOT.parent / "backend"
else:
    BACKEND = ROOT
sys.path.insert(0, str(BACKEND))

os.environ.setdefault("DB_HOST", "127.0.0.1")
os.environ.setdefault("DB_PORT", "3306")
os.environ.setdefault("DB_NAME_PROD", "PlayerDev")


def _find_gcloud() -> str:
    """Jupyter kernels often lack the interactive shell PATH; resolve gcloud.cmd."""
    found = shutil.which("gcloud") or shutil.which("gcloud.cmd")
    if found:
        return found
    candidates = [
        Path.home() / "AppData/Local/Google/Cloud SDK/google-cloud-sdk/bin/gcloud.cmd",
        Path(os.environ.get("LOCALAPPDATA", "")) / "Google/Cloud SDK/google-cloud-sdk/bin/gcloud.cmd",
        Path(os.environ.get("ProgramFiles", r"C:\\Program Files")) / "Google/Cloud SDK/google-cloud-sdk/bin/gcloud.cmd",
    ]
    for p in candidates:
        if p.is_file():
            return str(p)
    raise FileNotFoundError(
        "gcloud not found. Install Google Cloud SDK, or set DB_USER/DB_PASS in this cell manually."
    )


# Kernel does NOT see PowerShell env vars. Load Cloud Run DB creds from Secret Manager.
if (
    not os.environ.get("DB_USER")
    or not os.environ.get("DB_PASS")
    or os.environ.get("DB_USER") in ("your_user",)
):
    gcloud = _find_gcloud()
    print("Using gcloud at:", gcloud)

    def _secret(name: str) -> str:
        return subprocess.check_output(
            [
                gcloud, "secrets", "versions", "access", "latest",
                f"--secret={name}",
                "--project=norse-coral-441421-r9",
            ],
            text=True,
        ).strip()

    os.environ["DB_USER"] = _secret("DB_USER")
    os.environ["DB_PASS"] = _secret("DB_PASS")
    print("Loaded DB_USER/DB_PASS from Secret Manager")

# Reload db so kernel picks up latest SSL/proxy fix on disk
import db as _db_mod
importlib.reload(_db_mod)
from db import get_db_connection
import report_metrics

ASSESSMENT_ID = 3  # Jason Peele retest 2026-03-08
print("backend:", BACKEND)
print("DB_HOST:", os.environ.get("DB_HOST"), "DB_USER set:", bool(os.environ.get("DB_USER")))
print("DB_USER starts with:", (os.environ.get("DB_USER") or "")[:4] + "...")

Using gcloud at: C:\Users\skgoa\AppData\Local\Google\Cloud SDK\google-cloud-sdk\bin\gcloud.CMD
Loaded DB_USER/DB_PASS from Secret Manager
backend: C:\Users\skgoa\OneDrive\Documents\Code\HEAT-Assessment-Dates\backend
DB_HOST: 127.0.0.1 DB_USER set: True
DB_USER starts with: repl...


### Cell — load assessment + build metrics bundle

Connects to PlayerDev, loads the assessment row, then runs M2 Blast + HitTrax aggregates for that calendar day only.

In [3]:
conn = get_db_connection()
row = report_metrics._fetch_peer(conn, ASSESSMENT_ID)
assert row, f"assessment {ASSESSMENT_ID} not found"
print("player:", row["player_name"])
print("date:", row["assessment_date"], "type:", row.get("assessment_type"))
print("baseline_id:", row.get("baseline_assessment_id"), "previous_id:", row.get("previous_assessment_id"))

bundle = report_metrics.build_report_bundle(conn, row)
cur = bundle["current"]
print("window:", cur["start_ts"], "->", cur["end_ts"])
print("BLAST:", cur["blast"])
print("HITTRAX:", cur["hittrax"])

player: Jason Peele
date: 2026-03-08 type: retest
baseline_id: None previous_id: 2
window: 2026-03-08 00:00:00 -> 2026-03-08 23:59:59
BLAST: {'swing_count': 100, 'peak_bat_speed': 71.5, 'avg_bat_speed': 59.49, 'sd_bat_speed': 4.88, 'avg_attack_angle': 8.71, 'sd_attack_angle': 5.41}
HITTRAX: {'swing_count': 63, 'peak_ev': 84.08, 'avg_ev': 72.99, 'p90_ev': 81.63, 'avg_launch_angle': 18.6, 'avg_la_hard_hit': 15.75, 'avg_ev_ideal_la': 77.88, 'avg_distance': 161.08}


### Cell — compare current / previous / baseline

One-line summary for CURRENT / PREVIOUS / BASELINE sides of the report.

In [4]:
def side_summary(label, side):
    if not side:
        print(f"{label}: (none)")
        return
    print(
        f"{label}: id={side['assessment_id']} date={side.get('assessment_date')} "
        f"blast={side['blast']['swing_count']} hittrax={side['hittrax']['swing_count']} "
        f"peak_bat={side['blast'].get('peak_bat_speed')} peak_ev={side['hittrax'].get('peak_ev')}"
    )

side_summary("CURRENT", bundle["current"])
side_summary("PREVIOUS", bundle.get("previous"))
side_summary("BASELINE", bundle.get("baseline"))

CURRENT: id=3 date=2026-03-08 blast=100 hittrax=63 peak_bat=71.5 peak_ev=84.08
PREVIOUS: id=2 date=2026-01-12 blast=86 hittrax=47 peak_bat=74.0 peak_ev=81.43
BASELINE: id=1 date=2025-11-23 blast=0 hittrax=53 peak_bat=None peak_ev=86.99


### Cell — day-only sanity check

Asserts swing counts match and every row falls on the assessment calendar day.

In [5]:
import datetime as dt

player = row["player_name"]
start, end = report_metrics._window_bounds(row)
blast_rows = report_metrics._blast_rows(conn, player, start, end)
ht_rows = report_metrics._hittrax_rows(conn, player, start, end)

print(f"raw blast rows: {len(blast_rows)}  hittrax rows: {len(ht_rows)}")
assert len(blast_rows) == cur["blast"]["swing_count"]
assert len(ht_rows) == cur["hittrax"]["swing_count"]

aday = row["assessment_date"]
if not hasattr(aday, "strftime"):
    aday = dt.date.fromisoformat(str(aday)[:10])

def _day(ts):
    if isinstance(ts, dt.datetime):
        return ts.date()
    if isinstance(ts, dt.date):
        return ts
    return None

ht_off = sum(1 for r in ht_rows if _day(r.get("TS")) not in (None, aday))
blast_off = sum(1 for r in blast_rows if _day(r.get("created_date")) not in (None, aday))
print(f"day-only check: blast off-day={blast_off}  hittrax off-day={ht_off}")
assert ht_off == 0 and blast_off == 0, "rows found outside assessment calendar day"

raw blast rows: 100  hittrax rows: 63
day-only check: blast off-day=0  hittrax off-day=0


### Cell — quick charts (optional)

Histograms of Blast bat speed and HitTrax EV. Needs matplotlib.

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    print("matplotlib not installed — skip charts (pip install matplotlib)")
else:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    bat = [float(r["metric_bat_speed"]) for r in blast_rows if r.get("metric_bat_speed") is not None]
    axes[0].hist(bat, bins=15, color="#fcba39", edgecolor="#333")
    axes[0].set_title(f"Blast bat speed (n={len(bat)})")
    axes[0].set_xlabel("mph")

    ev = [float(r["ev"]) for r in ht_rows if r.get("ev") is not None]
    axes[1].hist(ev, bins=15, color="#0b3d5c", edgecolor="#eee")
    axes[1].set_title(f"HitTrax EV / EBV1 (n={len(ev)})")
    axes[1].set_xlabel("mph (converted)")

    plt.tight_layout()
    plt.show()

### Cell — draft PDF preview (M3)

Writes a local draft PDF from the same metrics bundle. Closes the DB connection when done.

In [7]:
import report_pdf

pdf_path = report_pdf.build_pdf(bundle)
print("Wrote", pdf_path)
conn.close()

Wrote reports\jason_peele_20260308_3.pdf
